In [58]:
"""
光伏/能耗代理模型 —— 训练完整流程（防泄漏版）
核心改动相对于旧版：
  1) 外层按 source_file（整栋楼）划分 train/test
  2) 内层超参搜索改用 GroupKFold
  3) 用 cross_val_predict(GroupKFold) 生成折外预测(OOF)
  4) 用 GES 基于 OOF 自动选模型 + 定权重
  5) 测试集只在最后评估一次
  6) 保留 WeightedEnsemble，并额外暴露 feature_names_in_
"""

import os
import time
import warnings
import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GridSearchCV, GroupKFold, cross_val_predict
from sklearn.base import clone
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

from sklearn.ensemble import (RandomForestRegressor, HistGradientBoostingRegressor)
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import Ridge, Lasso
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

from ensemble_utils import WeightedEnsemble

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

# ============================== 全局配置 ==============================
DATA_PATH = 'all_cleaned_data.csv'
SAVE_DIR = r'D:\research_paper\photovoltaic_prediction\training_results'
PLOT_DIR = os.path.join(SAVE_DIR, 'plot')
MODEL_DIR = os.path.join(SAVE_DIR, 'model')
for d in (SAVE_DIR, PLOT_DIR, MODEL_DIR):
    os.makedirs(d, exist_ok=True)

TARGET = 'hourly_EUI'

CV_FOLDS = 5
PARALLEL_JOBS = -3
GES_ITERS = 50            # GES 迭代次数（带放回）
GES_METRIC = 'rmse'       # 'rmse' 或 'mae'

# TabPFN 可选基线（数据量大时强烈建议保持 False，或仅子样本拟合）
USE_TABPFN = False
TABPFN_MAX_TRAIN = 10000  # TabPFN 子样本上限

FLOOR_HEIGHT = 3.0
PLOT_AREA = 62500.0


In [59]:
# ============================== 1. 预处理 ==============================
def preprocess_data(df):
    print("开始数据预处理与特征工程...")
    df = df.sort_values(['source_file', 'date']).reset_index(drop=True)

    df['main_type'] = df['building_type'].apply(lambda x: str(x).split('_')[0])
    df['code_classification'] = df['building_type'].apply(
        lambda x: "_".join(str(x).split('_')[1:]) if '_' in str(x) else 'Unknown')

    df['timestamp'] = pd.to_datetime(df['date'])
    df['hour'] = df['timestamp'].dt.hour
    df['month'] = df['timestamp'].dt.month
    df['day_of_year'] = df['timestamp'].dt.dayofyear

    def get_season(m):
        return ('spring' if m in (3, 4, 5) else 'summer' if m in (6, 7, 8)
                else 'autumn' if m in (9, 10, 11) else 'winter')
    df['season'] = df['month'].apply(get_season)

    df['is_daylight'] = (df['GHI'] > 0).astype(int)
    df['cos_solar_alt'] = np.cos(np.radians(df['Solar Altitude'].fillna(0).clip(lower=0)))

    # 形态物理特征
    a_roof = df['frontage'] * df['depth'] * df['Building Amount']
    a_wall_ew = 2.0 * (df['depth'] * df['Building Height'] * df['Building Amount'])
    a_wall_sn = 2.0 * (df['frontage'] * df['Building Height'] * df['Building Amount'])
    a_env = a_roof + a_wall_ew + a_wall_sn

    df['roof_to_envelope_ratio'] = a_roof / (a_env + 1e-9)
    floor_num = df['Building Height'] / FLOOR_HEIGHT
    df['roof_to_floor_ratio'] = 1.0 / (floor_num + 1e-9)
    footprint = df['frontage'] * df['depth'] * df['Building Amount']
    df['Building density'] = footprint / PLOT_AREA
    df['OSR'] = (PLOT_AREA - footprint) / PLOT_AREA

    df.drop([c for c in ['date'] if c in df.columns], axis=1, inplace=True)
    print(f"预处理完成。当前特征总数: {len(df.columns)}")

    return df

In [60]:
# ============================== 2. 划分（按楼分层） ==============================
def split_by_building(df, train_ratio=0.8):
    id_cols = ['source_file', 'main_type', 'code_classification']
    model_info = df[id_cols].drop_duplicates().copy()
    train_files, test_files = [], []
    for _, group in model_info.groupby('main_type'):
        n_train = int(round(len(group) * train_ratio))
        files = group.sort_values('source_file')['source_file'].tolist()
        train_files.extend(files[:n_train])
        test_files.extend(files[n_train:])
    print(f"建筑划分：训练 {len(train_files)} 栋 / 测试 {len(test_files)} 栋")
    return (df[df['source_file'].isin(train_files)].copy(),
            df[df['source_file'].isin(test_files)].copy())

In [61]:
# ============================== 3. 评估函数 ==============================
def evaluate_regress(y_pre, y_true, target_name, model_name, verbose=True):
    y_pre = np.asarray(y_pre, dtype=np.float64).ravel()
    y_true = np.asarray(y_true, dtype=np.float64).ravel()
    MAE = mean_absolute_error(y_true, y_pre)
    RMSE = np.sqrt(mean_squared_error(y_true, y_pre))
    R2_all = r2_score(y_true, y_pre)
    threshold = 0.001 if 'EUI' in target_name else 0.1
    mask = y_true > threshold
    R2_nz = r2_score(y_true[mask], y_pre[mask]) if np.any(mask) else R2_all
    if verbose:
        print(f'>>> {model_name} | R2(all): {R2_all:.4f} | R2(nonzero): {R2_nz:.4f} | MAE: {MAE:.4f}')
    return MAE, RMSE, R2_all, R2_nz

In [62]:
# ============================== 4. 单模型训练（GroupKFold 调参） ==============================
def train_model(X, y, groups, method, gkf):
    """返回在全训练集上拟合好的最优模型。所有 GridSearchCV 使用 GroupKFold + groups。"""
    if method == 'LightGBM':
        est = LGBMRegressor(objective='regression', random_state=42, n_jobs=1, verbose=-1)
        g1 = GridSearchCV(est, {'n_estimators': [300], 'learning_rate': [0.1],
                                'colsample_bytree': [0.3, 0.5], 'num_leaves': [15, 31]},
                          cv=gkf, scoring='r2', n_jobs=PARALLEL_JOBS)
        g1.fit(X, y, groups=groups)
        est2 = LGBMRegressor(**g1.best_params_, objective='regression',
                             random_state=42, n_jobs=1, verbose=-1)
        g2 = GridSearchCV(est2, {'min_child_samples': [2000, 5000], 'reg_lambda': [500, 1000]},
                          cv=gkf, scoring='r2', n_jobs=PARALLEL_JOBS)
        g2.fit(X, y, groups=groups)
        return g2.best_estimator_

    if method == 'XGBoost':
        est = XGBRegressor(random_state=42, tree_method='hist', n_jobs=1)
        g1 = GridSearchCV(est, {'n_estimators': [200, 300], 'max_depth': [3, 4],
                                'colsample_bytree': [0.2, 0.4]},
                          cv=gkf, scoring='r2', n_jobs=PARALLEL_JOBS)
        g1.fit(X, y, groups=groups)
        est2 = XGBRegressor(**g1.best_params_, random_state=42, tree_method='hist', n_jobs=1)
        g2 = GridSearchCV(est2, {'min_child_weight': [100, 200], 'reg_lambda': [1000]},
                          cv=gkf, scoring='r2', n_jobs=PARALLEL_JOBS)
        g2.fit(X, y, groups=groups)
        return g2.best_estimator_

    if method == 'CatBoost':
        est = CatBoostRegressor(random_state=42, verbose=0, thread_count=1, learning_rate=0.1)
        g1 = GridSearchCV(est, {'iterations': [200], 'depth': [4, 5], 'colsample_bylevel': [0.3]},
                          cv=gkf, scoring='r2', n_jobs=PARALLEL_JOBS)
        g1.fit(X, y, groups=groups)
        est2 = CatBoostRegressor(**g1.best_params_, random_state=42, verbose=0,
                                 thread_count=1, learning_rate=0.1)
        g2 = GridSearchCV(est2, {'l2_leaf_reg': [500, 1000]},
                          cv=gkf, scoring='r2', n_jobs=PARALLEL_JOBS)
        g2.fit(X, y, groups=groups)
        return g2.best_estimator_

    if method == 'RF':
        est = RandomForestRegressor(random_state=42, n_jobs=1, max_samples=0.1)
        g = GridSearchCV(est, {'n_estimators': [100], 'max_depth': [10],
                               'min_samples_leaf': [50, 100]},
                         cv=gkf, scoring='r2', n_jobs=PARALLEL_JOBS)
        g.fit(X, y, groups=groups)
        return g.best_estimator_

    if method == 'GBR':
        est = HistGradientBoostingRegressor(random_state=42, early_stopping=True)
        g = GridSearchCV(est, {'max_iter': [300], 'l2_regularization': [50.0, 150.0]},
                         cv=gkf, scoring='r2', n_jobs=PARALLEL_JOBS)
        g.fit(X, y, groups=groups)
        return g.best_estimator_

    if method == 'DT':
        g = GridSearchCV(DecisionTreeRegressor(), {'max_depth': [8], 'min_samples_leaf': [50]},
                         cv=gkf, scoring='r2', n_jobs=PARALLEL_JOBS)
        g.fit(X, y, groups=groups)
        return g.best_estimator_

    if method == 'Ridge':
        g = GridSearchCV(Ridge(), {'alpha': [100, 1000]}, cv=gkf, scoring='r2', n_jobs=PARALLEL_JOBS)
        g.fit(X, y, groups=groups)
        return g.best_estimator_

    if method == 'Lasso':
        g = GridSearchCV(Lasso(), {'alpha': [1.0, 10.0]}, cv=gkf, scoring='r2', n_jobs=PARALLEL_JOBS)
        g.fit(X, y, groups=groups)
        return g.best_estimator_

    raise ValueError(f"未知模型: {method}")

In [63]:
# ============================== 5. GES 贪心集成选择 ==============================
def greedy_ensemble_selection(oof_preds, y_true, n_iter=50, metric='rmse'):
    """Caruana(2004) 带放回贪心集成选择，基于折外预测(OOF)，无泄漏。
    oof_preds: {name: 1D array}；返回 {name: weight}（被选中次数归一化）。"""
    names = list(oof_preds)
    P = {n: np.asarray(oof_preds[n], dtype=np.float64).ravel() for n in names}
    y = np.asarray(y_true, dtype=np.float64).ravel()
    if metric == 'rmse':
        err = lambda p: np.sqrt(np.mean((p - y) ** 2))
    else:
        err = lambda p: np.mean(np.abs(p - y))

    counts = {n: 0 for n in names}
    ens_sum = np.zeros_like(y)
    k = 0
    history = []
    for _ in range(n_iter):
        best_n, best_e = None, np.inf
        for n in names:
            e = err((ens_sum + P[n]) / (k + 1))
            if e < best_e:
                best_e, best_n = e, n
        ens_sum += P[best_n]
        counts[best_n] += 1
        k += 1
        history.append(best_e)
    total = sum(counts.values())
    weights = {n: counts[n] / total for n in names if counts[n] > 0}
    return weights, history

In [64]:
# ============================== 6. 加权集成（与优化脚本兼容） ==============================
class WeightedEnsemble:
    """加权平均集成。额外暴露 feature_names_in_ 以便优化脚本读取特征顺序。"""
    def __init__(self, models, weights, feature_names=None):
        self.models = list(models)
        self.weights = list(weights)
        if feature_names is not None:
            self.feature_names_in_ = np.asarray(feature_names, dtype=object)
            self.feature_name_ = list(feature_names)   # LightGBM 风格别名
        else:
            self.feature_names_in_ = None
            self.feature_name_ = None

    def predict(self, X):
        preds = [np.asarray(m.predict(X)).ravel() for m in self.models]
        return np.average(preds, axis=0, weights=self.weights)

In [65]:
# ============================== 主流程 ==============================
def main():
    # ---- 加载 + 预处理 ----
    df = pd.read_csv(DATA_PATH)
    df = preprocess_data(df)

    train_df, test_df = split_by_building(df, train_ratio=0.8)

    # 标签编码（在全量类别上 fit 词表，避免测试集出现未见类别报错）
    le_main, le_code = LabelEncoder(), LabelEncoder()
    le_main.fit(df['main_type'])
    le_code.fit(df['code_classification'])
    for d in (train_df, test_df):
        d['main_type_encoded'] = le_main.transform(d['main_type'])
        d['code_encoded'] = le_code.transform(d['code_classification'])

    # 季节独热
    for d_name, d in (('train', train_df), ('test', test_df)):
        dummies = pd.get_dummies(d['season'], prefix='season').astype(np.int8)
        for col in dummies.columns:
            d[col] = dummies[col].values
        d.drop('season', axis=1, inplace=True)

    # ---- 特征矩阵（注意：列名与 CSV 一致，含空格；优化脚本须用相同名字）----
    feature_cols = [
        'FAR', 'Building density', 'Building Height', 'Building Amount', 'Shape Factor',
        'depth', 'frontage', 'rotation',
        'SVF', 'roof_to_envelope_ratio', 'roof_to_floor_ratio', 'OSR',
        'dry_bulb_temperature', 'relative_humidity', 'DNI', 'DHI', 'GHI',
        'hour', 'month', 'day_of_year', 'is_daylight', 'cos_solar_alt',
        'main_type_encoded', 'code_encoded'
    ]
    # 保留 DataFrame（不转 .values），让各模型记录 feature_names_in_
    X_train = train_df[feature_cols].astype(np.float32)
    y_train = train_df[TARGET].astype(np.float32)
    X_test = test_df[feature_cols].astype(np.float32)
    y_test = test_df[TARGET].astype(np.float32)
    groups_train = train_df['source_file'].values   # 与 X_train 行对齐

    print(f"特征维度: {len(feature_cols)} | 训练 {X_train.shape} | 测试 {X_test.shape}")

    gkf = GroupKFold(n_splits=CV_FOLDS)

    # ---- 训练各基模型 + 生成 OOF ----
    name_list = ['LightGBM', 'XGBoost', 'CatBoost', 'RF', 'DT', 'GBR', 'Ridge', 'Lasso']
    fitted_models, success, oof_preds = {}, [], {}
    train_rows, test_rows = [], []

    for method in name_list:
        print(f"\n====== 训练: {method} ======")
        t0 = time.time()
        try:
            model = train_model(X_train, y_train, groups_train, method, gkf)
            print(f"  完成，耗时 {time.time()-t0:.1f}s。计算 OOF...")
            # 用相同配置在 GroupKFold 上做折外预测（无泄漏）
            oof = cross_val_predict(clone(model), X_train, y_train,
                                    cv=gkf, groups=groups_train, n_jobs=PARALLEL_JOBS)
            oof_preds[method] = oof
            fitted_models[method] = model
            success.append(method)

            m_oof = evaluate_regress(oof, y_train, TARGET, method + '_OOF')
            m_test = evaluate_regress(model.predict(X_test), y_test, TARGET, method + '_Test')
            train_rows.append(m_oof)
            test_rows.append(m_test)
        except Exception as e:
            print(f"  ❌ {method} 失败: {e}")

    cols = ['MAE', 'RMSE', 'R2', 'R2_Nonzero']
    oof_df = pd.DataFrame(train_rows, columns=cols, index=success)   # 注意：这是 OOF，不是训练集 in-sample
    test_df_results = pd.DataFrame(test_rows, columns=cols, index=success)
    print("\n=== 各基模型 OOF 表现（用于选模型）===")
    print(oof_df.sort_values('R2', ascending=False))

    # ---- GES：基于 OOF 自动选模型 + 定权重 ----
    if len(success) >= 2:
        weights, _ = greedy_ensemble_selection(oof_preds, y_train,
                                               n_iter=GES_ITERS, metric=GES_METRIC)
        print(f"\n✅ GES 选出的模型与权重: {weights}")

        sel_models = [fitted_models[n] for n in weights]   # 已在全训练集拟合
        sel_weights = list(weights.values())
        ensemble = WeightedEnsemble(sel_models, sel_weights, feature_names=feature_cols)

        # 测试集只在这里评估一次
        ens_metrics = evaluate_regress(ensemble.predict(X_test), y_test, TARGET, "Ensemble_GES")
        test_df_results.loc['Ensemble_Best'] = ens_metrics
        best_model, best_name = ensemble, 'Ensemble_Best'
    else:
        print("⚠️ 成功模型不足 2 个，退化为单最优模型。")
        best_name = oof_df['R2'].idxmax()
        best_model = fitted_models[best_name]

    # ---- 可选：TabPFN 作为最新 SOTA 基线（默认关闭）----
    if USE_TABPFN:
        try:
            from tabpfn import TabPFNRegressor
            print("\n[TabPFN] 作为对照基线，在训练子样本上拟合...")
            n_sub = min(TABPFN_MAX_TRAIN, len(X_train))
            idx = np.random.RandomState(42).choice(len(X_train), n_sub, replace=False)
            tab = TabPFNRegressor()
            tab.fit(X_train.iloc[idx].values, y_train.iloc[idx].values)
            evaluate_regress(tab.predict(X_test.values), y_test, TARGET, "TabPFN_baseline")
        except Exception as e:
            print(f"[TabPFN] 跳过（{e}）")

    print("\n" + "=" * 60)
    print("🏆 测试集评估汇总（按 R2 降序）")
    print(test_df_results.sort_values('R2', ascending=False))
    print("=" * 60)

    # ---- 保存 ----
    oof_df.to_csv(os.path.join(SAVE_DIR, f'model_oof_performance_{TARGET}.csv'))
    test_df_results.to_csv(os.path.join(SAVE_DIR, f'model_test_performance_{TARGET}.csv'))
    from joblib import dump
    dump(best_model, os.path.join(MODEL_DIR, f'best_model_{best_name}_{TARGET}.joblib'))
    print(f"最佳模型已保存: best_model_{best_name}_{TARGET}.joblib")

    # ---- 可视化 + SHAP ----
    try:
        make_plots(fitted_models, success, best_model, best_name,
                   X_test, y_test, oof_df)
    except Exception as e:
        print(f"绘图阶段异常（不影响模型保存）: {e}")

    return best_model, best_name, test_df_results

In [66]:
# ============================== 7. 可视化 + SHAP ==============================
def make_plots(fitted_models, success, best_model, best_name, X_test, y_test, oof_df):
    import matplotlib.pyplot as plt
    import seaborn as sns
    plt.style.use('seaborn-v0_8-whitegrid')
    plt.rcParams.update({'font.family': 'serif', 'font.serif': ['Times New Roman'],
                         'axes.unicode_minus': False})

    def scatter(y_true, y_pred, name, sample=10000):
        y_true = np.asarray(y_true).ravel(); y_pred = np.asarray(y_pred).ravel()
        if len(y_true) > sample:
            i = np.random.RandomState(0).choice(len(y_true), sample, replace=False)
            y_true, y_pred = y_true[i], y_pred[i]
        plt.figure(figsize=(8, 8))
        plt.scatter(y_true, y_pred, alpha=0.15, s=8, color='#2c3e50', edgecolors='none')
        lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
        plt.plot(lims, lims, 'r--', lw=2, label='Ideal (y=x)')
        plt.xlabel('Measured'); plt.ylabel('Predicted'); plt.title(name)
        plt.legend(); plt.tight_layout()
        plt.savefig(os.path.join(PLOT_DIR, f'scatter_{name}.png'), dpi=300, bbox_inches='tight')
        plt.close()

    preds_map = {}
    for m in success:
        p = fitted_models[m].predict(X_test)
        if not (np.isnan(p).any() or np.isinf(p).any()):
            preds_map[m] = p
            scatter(y_test, p, m + '_Test')
    if best_name == 'Ensemble_Best':
        pe = best_model.predict(X_test)
        preds_map['Ensemble_Best'] = pe
        scatter(y_test, pe, 'Ensemble_Best_Test')

    # 误差箱型 + 抖动散点
    if preds_map:
        order = (['Ensemble_Best'] + [m for m in preds_map if m != 'Ensemble_Best']
                 if 'Ensemble_Best' in preds_map else list(preds_map))
        rows = []
        yt = np.asarray(y_test).ravel()
        for name, pred in preds_map.items():
            ae = np.abs(yt - np.asarray(pred).ravel())
            d = pd.DataFrame({'Absolute Error': ae, 'Model': name})
            rows.append(d.sample(min(120, len(d)), random_state=42))
        err_df = pd.concat(rows, ignore_index=True)
        plt.figure(figsize=(14, 8))
        sns.boxplot(x='Model', y='Absolute Error', data=err_df, order=order,
                    showfliers=False, width=0.4, color='white', linewidth=1.5)
        sns.stripplot(x='Model', y='Absolute Error', data=err_df, order=order,
                      size=5, alpha=0.55, jitter=0.22,
                      palette=sns.color_palette("husl", len(order)))
        plt.xticks(rotation=45, ha='right'); plt.tight_layout()
        plt.savefig(os.path.join(PLOT_DIR, 'error_distribution_comparison.png'), dpi=300)
        plt.close()

    # SHAP（解释 OOF 表现最好的单体树模型）
    try:
        import shap
        best_single = oof_df['R2'].idxmax()
        mdl = fitted_models[best_single]
        explainer = shap.TreeExplainer(mdl)
        Xs = X_test.sample(min(2000, len(X_test)), random_state=42)
        sv = explainer.shap_values(Xs, check_additivity=False)
        sv = sv[1] if isinstance(sv, list) and len(sv) == 2 else sv

        plt.figure(figsize=(12, 9))
        shap.summary_plot(sv, Xs, plot_type='bar', show=False)
        plt.title(f'Global Feature Importance (SHAP): {best_single}')
        plt.tight_layout()
        plt.savefig(os.path.join(PLOT_DIR, f'shap_bar_{best_single}.png'), dpi=300, bbox_inches='tight')
        plt.close()

        plt.figure(figsize=(12, 9))
        shap.summary_plot(sv, Xs, show=False, alpha=0.6)
        plt.title(f'SHAP Beeswarm: {best_single}')
        plt.tight_layout()
        plt.savefig(os.path.join(PLOT_DIR, f'shap_beeswarm_{best_single}.png'), dpi=300, bbox_inches='tight')
        plt.close()
        print("  SHAP 图已保存。")
    except Exception as e:
        print(f"  SHAP 跳过: {e}")

    print(f"可视化已保存至: {PLOT_DIR}")


if __name__ == '__main__':
    main()

开始数据预处理与特征工程...
预处理完成。当前特征总数: 35
建筑划分：训练 675 栋 / 测试 168 栋
特征维度: 24 | 训练 (5913000, 24) | 测试 (1471680, 24)

====== 训练: LightGBM ======
  完成，耗时 290.8s。计算 OOF...
>>> LightGBM_OOF | R2(all): 0.9900 | R2(nonzero): 0.9874 | MAE: 0.0005
>>> LightGBM_Test | R2(all): 0.9892 | R2(nonzero): 0.9865 | MAE: 0.0005

====== 训练: XGBoost ======
  完成，耗时 311.6s。计算 OOF...
>>> XGBoost_OOF | R2(all): 0.9875 | R2(nonzero): 0.9846 | MAE: 0.0005
>>> XGBoost_Test | R2(all): 0.9869 | R2(nonzero): 0.9839 | MAE: 0.0006

====== 训练: CatBoost ======
  完成，耗时 635.6s。计算 OOF...
>>> CatBoost_OOF | R2(all): 0.9645 | R2(nonzero): 0.9558 | MAE: 0.0009
>>> CatBoost_Test | R2(all): 0.9632 | R2(nonzero): 0.9545 | MAE: 0.0009

====== 训练: RF ======
  完成，耗时 917.8s。计算 OOF...
>>> RF_OOF | R2(all): 0.9416 | R2(nonzero): 0.9239 | MAE: 0.0011
>>> RF_Test | R2(all): 0.9386 | R2(nonzero): 0.9202 | MAE: 0.0011

====== 训练: DT ======
  完成，耗时 64.0s。计算 OOF...
>>> DT_OOF | R2(all): 0.8963 | R2(nonzero): 0.8655 | MAE: 0.0015
>>> DT_Test | R2(all)